# 00 – Download Remote Sensing Data
**Purpose:** Download different datasets, which will be used in the following notebooks. To Download all the datasets around **.... GB/MB** of space are needed.

| Dataset | Working-Name | Resolution | Time-range | API | URL |
| --- | --- | --- | --- | --- |--- |
|SWOT_L2_HR_RiverSP_reach_D || global,  | Dec 2022 - July 2023: fast-sampling phase, Dec 2023 - ... beta/ pre-validated SWOT, Mar 2024 - present: C SWOT | Yes |https://swot.jpl.nasa.gov |
| ESA WorldCover ||  global, 10m | 2020 - 2021 |Yes |https://esa-worldcover.org/en|
|GlobalDamWatch||global||||
|||||||
|||||||

**Workflow:**
0 | Imports
- 0.1 | Configuration
- 0.2 | Hydrocron API (no login required)
- 0.3 | earthaccess (NASA login required)

**References:**
- SWOT Mission: https://swot.jpl.nasa.gov
- Hydrocron API: https://github.com/podaac/hydrocron
- PO.DAAC SWOT Data: (https://podaac.jpl.nasa.gov/SWOT)
- Tebaldi, N., Nickels, C. (2025): [*Hydrocron API: SWOT Time Series Examples.*](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples.html)
- Nickels, C. (2025): [*Search and Download SWOT Data via <code>earthaccess</code>*. ](https://podaac.github.io/tutorials/notebooks/SearchDownload_SWOTviaCMR.html)
- earthaccess docs: (https://earthaccess.readthedocs.io)
- NASA Earthdata Login: (https://urs.earthdata.nasa.gov)

To-Do:
- [] Thinking about variables which can be calculated by SWOT data and which are not in SWORD dataset
- [] making the workflow for investigation and download and variable calculation before download smarter.
- [] do I even need to download? I could just ask API for values, calculate new ones and join them to the SWORD dataset?
- [] How should can I besser handle the amount of AWOT data? Maybe...by..maybe creating a Goal-Reach_list and using parquet instead of shp and gpkg.
- [] adding Klimaclassification köppen & Geiger
- [x] Nochmal SWORD anschauen, ob es da auhc gute strahler-ordnung gibt.
-->     reach_id format: CBBBBBRRRRT
    - C      : Continent (1 digit)
    - BBBBB  : Pfafstetter basin code up to level 6 (5 digits)
    - RRR    : Reach number within basin (3 digits)
    - T      : Type (1 digit)
    
    Returns: str - first 6 digits of reach_id
- [] Connectivity Index in River Networks für die GDW (aufbauend, also pro z.B. Damm immer nur Oberfluss)
- [] Wegen GDW: Schauen, ob sich Messwerte in SWORD nodes signifikant an der Stelle ändern, an welcher das GDW Objekt auftritt.

- [] connectivity flussaufwärts mit GDW bauwerke 

---
---
## 0 | Imports


In [ ]:
import geopandas as gpd
import os
import sys
import earthaccess
import rasterio
import zipfile
import requests
from pathlib import Path
from soilgrids import soilgrids

sys.path.append(os.path.join("..", "src"))

from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT, IN_SWORD
from _00_config import STUDY_AREA, PFAF_IDS
from _00_setup import setup_study_area
from _0_config_paths import IN_BASIN_5
from _00_config import PFAF_IDS, PFAF_LEVEL_DIGITS

from _00_download_rs_data import (
    query_hydrocron_bulk,
    parse_hydrocron_results,
    compute_bbox,
    search_swot_granules,
    stream_swot_granules,
    download_soilgrids,
    download_worldcover,
    merge_raster_tiles,
    download_opentopo_dem,
    download_gloric_v10,
    download_gdw
)

SAGA found: C:\Program Files\QGIS 3.42.3\apps\saga7\saga_cmd.exe


---
---
## 0.1 | Configuration

In [ ]:
# ============================================================
# Study area is defined in src/_00_config.py
# Only download-specific settings are configured here
# ============================================================

# Load AOI and SWORD from study area setup
aoi, sword = setup_study_area(
    basin_path        = IN_BASIN_5,
    sword_path        = IN_SWORD,
    pfaf_ids          = PFAF_IDS,
    pfaf_level_digits = PFAF_LEVEL_DIGITS
)
bbox = tuple(aoi.total_bounds)
print(f"AOI bbox: {bbox}")

# OpenTopography API key (set as environment variable, never hardcode)
OPENTOPO_API_KEY = os.environ.get("OPENTOPO_API_KEY", "")

#--- TIME RANGE for SWOT observations --------------------------------
START_TIME = "2025-08-01T00:00:00Z"
END_TIME   = "2025-08-31T00:00:00Z"

#--- HYDROCRON settings -----------------------------------------------
HYDROCRON_FIELDS        = "reach_id,time_str,wse,slope,width,p_dist_out,geometry"
HYDROCRON_OUTPUT_FORMAT = "geojson"
HYDROCRON_REQUEST_DELAY = 0.3
MIN_WIDTH_M             = 100

#--- EARTHACCESS settings ---------------------------------------------
EARTHACCESS_COLLECTION = "SWOT_L2_HR_RiverSP_reach_D"
GRANULE_STRIDE         = 1
MAX_GRANULES           = None

#--- OUTPUT paths -------------------------------------------------------
OUT_HYDROCRON     = os.path.join(DATA_PROCESSED, f"swot_hydrocron_{STUDY_AREA}.gpkg")
OUT_EARTHACCESS   = os.path.join(DATA_PROCESSED, f"swot_earthaccess_{STUDY_AREA}.gpkg")
OUT_WORLDCOVER_DIR = os.path.join(DATA_RAW, "0_data", "worldcover")
OUT_WORLDCOVER_MERGED = os.path.join(DATA_RAW, "0_data", "worldcover", "worldcover_merged.tif")
OUT_SOILGRIDS_DIR = os.path.join(DATA_RAW, "0_data", "soilgrids")
OUT_COP30_DEM     = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example",
                                  f"{STUDY_AREA}_cop30_dem.tif")
OUT_GDW_DIR       = os.path.join(DATA_RAW, "0_data", "GDW")
OUT_GLORIC_V10    = os.path.join(DATA_RAW, "0_data", "GloRiC_v10.gpkg")

SOILGRIDS_VARS = [
    ("clay", "clay_0-5cm_mean", "Clay content (%) – cohesive banks"),
    ("sand", "sand_0-5cm_mean", "Sand content (%) – non-cohesive banks"),
    ("silt", "silt_0-5cm_mean", "Silt content (%) – intermediate grain size"),
    ("cfvo", "cfvo_0-5cm_mean", "Coarse fragments (%) – rocky soils"),
]

print(f"Study area : {STUDY_AREA}")
print(f"PFAF_IDs   : {PFAF_IDS}")
print(f"bbox       : {bbox}")

---
---
## 0.2 | Hydrocron API

Queries SWOT time series per SWORD reach. No file download, no NASA login required.
SWOT observes rivers wider than ca. 100 m, narrower reaches are skipped automatically.

Needing the **Hydrocron API** (PO.DAAC / NASA JPL) for each SWORD reach in the AOI.
Returns all SWOT observations within the configured time range as GeoJSON or CSV.

**How:** SWOT data is archived as globally-tiled shapefiles per satellite pass
(one file = one pass over a whole continent). Hydrocron unpacks these shapefiles and
re-indexes them by reach_id, so a single API call returns all observations for one reach
across all passes --> without downloading any file. **No NASA login required.**

https://github.com/podaac/hydrocron

In [ ]:
print(f"SWORD reaches loaded: {len(sword)} | CRS: {sword.crs}")

# Query all reaches via Hydrocron API
all_features = query_hydrocron_bulk(
    sword = sword,
    start_time= START_TIME,
    end_time = END_TIME,
    fields = HYDROCRON_FIELDS,
    min_width_m = MIN_WIDTH_M,
    request_delay = HYDROCRON_REQUEST_DELAY
)

In [ ]:
# Parse raw GeoJSON into clean GeoDataFrame and save
swot_gdf = parse_hydrocron_results(all_features)

if swot_gdf is not None:
    num_cols = [c for c in ["wse", "slope", "width"] if c in swot_gdf.columns]
    print(swot_gdf[num_cols].describe().round(3))
    print(f"\nObservations per reach:")
    print(swot_gdf.groupby("reach_id").size().describe())

    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_gdf.to_file(OUT_HYDROCRON, driver="GPKG")
    print(f"\nSaved: {OUT_HYDROCRON}")

---
---
## 0.3 | earthaccess – Download SWOT granules

Downloads the actual SWOT Shapefile granules (one file = one satellite pass over a
continent). Files are filtered spatially by bounding box and temporally by date range.

**downloadable in this section:**
- Raster products (`L2_HR_Raster`)
- Full Shapefile attributes (not just the Hydrocron field subset)

**!!!Warning:!!!** Each SWOT granule covers an entire continent pass and can be **several
hundred MB to several GB**. After download the data is clipped to the AOI.

**NASA Earthdata Login required.** acc at: https://urs.earthdata.nasa.gov

References:
- earthaccess: https://earthaccess.readthedocs.io
- SWOT collections: https://podaac.github.io/tutorials/quarto_text/SWOT.html
- PO.DAAC Cookbook: https://podaac.github.io/tutorials

In [ ]:
# EARTHACCESS: authenticate with NASA Earthdata Login

# earthaccess.login() will:
#   Try to use a stored .netrc file silently
#   If not found: prompt for username + password interactively

auth = earthaccess.login(strategy="interactive", persist=True)
print(f"Login successful: {auth.authenticated}")

In [ ]:
# Search for matching granules
results = search_swot_granules(
    collection = EARTHACCESS_COLLECTION,
    bbox = bbox,
    start_time = START_TIME,
    end_time = END_TIME
)

# Stream granules in-memory, clip to AOI, merge into GeoDataFrame
swot_ea = stream_swot_granules(
    results = results,
    aoi_bounds = sword.total_bounds,
    max_granules = MAX_GRANULES,
    granule_stride = GRANULE_STRIDE
)

if swot_ea is not None:
    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_ea.to_file(OUT_EARTHACCESS, driver="GPKG")
    print(f"Saved: {OUT_EARTHACCESS}")

---
## 0.4 | Download ESA WorldCover

In [5]:
# sword and bbox were already loaded in Section 0.2
worldcover_files = download_worldcover(
    bbox    = bbox,
    out_dir = OUT_WORLDCOVER_DIR
)
print(f"WorldCover tiles: {worldcover_files}")

Tiles needed: 8
  lon=69, lat=39
  lon=69, lat=42
  lon=72, lat=39
  lon=72, lat=42
  lon=75, lat=39
  lon=75, lat=42
  lon=78, lat=39
  lon=78, lat=42
Downloading: ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N39E072_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E072_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E072_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E072_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N39E075_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E075_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E075_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E07

In [6]:
worldcover_merged = merge_raster_tiles(
    tile_paths = worldcover_files,
    out_path   = OUT_WORLDCOVER_MERGED
)

Merging 8 tiles...
Merged 8 tiles → D:\0_InnoLab\0_data\WorldCover\WorldCover_merged.tif


---
## 0.5 | Download soilgrids


In [ ]:
soilgrids_files = download_soilgrids(
    bbox           = bbox,
    out_dir        = OUT_SOILGRIDS_DIR,
    soilgrids_vars = SOILGRIDS_VARS
)
print(f"soilgrids files: {soilgrids_files}")

Requesting grid: 2927 x 671 pixels
Downloading: clay_0-5cm_mean – Clay content (%) – cohesive banks, meandering tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\clay_0-5cm_mean.tif
Downloading: sand_0-5cm_mean – Sand content (%) – non-cohesive banks, braided tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\sand_0-5cm_mean.tif
Downloading: silt_0-5cm_mean – Silt content (%) – intermediate grain size
  Saved: D:\0_InnoLab\0_data\SoilGrids\silt_0-5cm_mean.tif
Downloading: cfvo_0-5cm_mean – Coarse fragments volume (%) – rocky soils, mountain reaches
  Saved: D:\0_InnoLab\0_data\SoilGrids\cfvo_0-5cm_mean.tif

Download complete: 4 variables
SoilGrids files: ['D:\\0_InnoLab\\0_data\\SoilGrids\\clay_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\sand_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\silt_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\cfvo_0-5cm_mean.tif']


---
## 06 | Download MERIT DEM 

In [ ]:
dem_path = download_opentopo_dem(
    bbox=bbox,
    out_path=OUT_COP30_DEM,
    api_key=OPENTOPO_API_KEY,
    demtype="COP30"
)

Saved: D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem.tif


In [6]:
print(OUT_COP30_DEM)

D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem.tif


In [ ]:
import rasterio
with rasterio.open(OUT_COP30_DEM) as src:
    print(f"Shape: {src.shape}")
    print(f"CRS: {src.crs}")
    # Read only a small portion to test
    data = src.read(1, window=rasterio.windows.Window(0, 0, 100, 100))
    print(f"Test read OK - min: {data.min()}, max: {data.max()}")

CRS: EPSG:4326
Resolution: (0.0002777777777777778, 0.0002777777777777778)
Shape: (6161, 24456)
Bounds: BoundingBox(left=71.60458332222223, bottom=40.75458334444444, right=78.39791665555556, top=42.46597223333333)
Elevation range: 386m - 5875m
NoData value: None


---
## 07 | Download GlobalDamWatch (GDW)

Download the global dataset once.

In [4]:
gdw_dir = download_gdw(out_dir=OUT_GDW_DIR)
print(f"GDW ready at: {gdw_dir}")

NameError: name 'download_gdw' is not defined

---
## 08 | Download Global River Classification (GloRiC)

https://www.hydrosheds.org/products/gloric

In [3]:
gloric_gpkg = download_gloric_v10(gpkg_path=OUT_GLORIC_V10)

print(f"GloRiC v1.0 ready at: {gloric_gpkg}")

GloRiC v1.0 already exists, skipping: D:\0_InnoLab\0_data\GloRiC_v10.gpkg
GloRiC v1.0 ready at: D:\0_InnoLab\0_data\GloRiC_v10.gpkg
